# Portfolio Data Collections

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf

In [60]:
import pandas as pd

# Create the portfolio data
trump_tickers = {
    'Ticker': ['NVDA', 'TSLA', 'AAPL', 'BA', 'XOM', 'PLTR', 'JPM', 'DELL', 
               'BLK', 'GS', 'INTC', 'MU', 'MA', 'V', 'QCOM'],
    'Company': ['NVIDIA', 'Tesla', 'Apple', 'Boeing', 'ExxonMobil', 'Palantir', 
                'JPMorgan Chase & Co.', 'Dell Technologies', 'BlackRock', 
                'Goldman Sachs', 'Intel', 'Micron', 'Mastercard', 'Visa', 'Qualcomm'],
    'Weight': [10.0, 9.0, 8.5, 7.5, 7.0, 6.5, 6.0, 6, 7.0, 7.5, 7.5, 4.5, 4.5, 4.5, 4.0]
}

# Create DataFrame
ticker_data = pd.DataFrame(trump_tickers)
ticker_data['Ticker']

0     NVDA
1     TSLA
2     AAPL
3       BA
4      XOM
5     PLTR
6      JPM
7     DELL
8      BLK
9       GS
10    INTC
11      MU
12      MA
13       V
14    QCOM
Name: Ticker, dtype: object

In [82]:
def portfolio_analytics(tickers, risk_free_rate, n_trading_days, confidence_level,weights = None, port_type = None):
    import yfinance as yf
    import pandas as pd
    import numpy as np
    rng = np.random.default_rng(1000)
    
    # Loading the data 
    df_close = yf.download(tickers=tickers, start='2020-01-01', auto_adjust=True)['Close']
    df_close.columns = tickers
    df_vol = yf.download(tickers=tickers, start='2020-01-01', auto_adjust=True)['Volume']
    df_vol.columns = tickers
    df_mcap = df_close * df_vol


    # Wrangling the data
    drop_10na_cols = df_close.columns[df_close.isna().sum() > 200]
    df_close = df_close.drop(columns=drop_10na_cols)

    # Log returns
    df_returns = np.log(df_close / df_close.shift(1)).dropna()
    n_assets = df_returns.shape[1]
    
    # Weighting techniques
    if port_type == 'EW':
        weights = np.ones(n_assets) / n_assets
    elif port_type == 'MC':
        weights = df_mcap.iloc[-1:,].values / df_mcap.iloc[-1:,].sum(axis = 1).values
    elif port_type == 'RP':
        weights = df_returns.std().values
        weights /= weights.sum()
    elif port_type == 'CW':
        if weights is None:
            raise ValueError("For port_type='CW', you must provide weights parameter")
        weights = np.array(weights).flatten()
        weights = weights / weights.sum()

    else:
        weights = rng.normal(size=n_assets)
        weights /= weights.sum()


    # Portfolio Returns
    cov_mat = df_returns.cov()
    port_returns =  df_returns @ weights
    port_variance = weights.T @ cov_mat @ weights

    # Annualized Returns
    annual_returns = port_returns.mean() * n_trading_days
    annual_std = np.sqrt(port_variance * n_trading_days)
    annual_sharpe = (annual_returns - risk_free_rate) / annual_std
    daily_var = np.percentile(port_returns, (1 - confidence_level) * 100)
    annual_var = daily_var * np.sqrt(n_trading_days)
    daily_es = port_returns[port_returns <= daily_var].mean()
    annual_es = daily_es * np.sqrt(n_trading_days)

    annual_results = pd.DataFrame({
        'Annual Port Return': [annual_returns],
        'Annual Port Std': [annual_std],
        'Annual Port Sharpe': [annual_sharpe],
        f'Annual VaR_{confidence_level}': [annual_var],
        f'Annual ES_{confidence_level}': [annual_es]
    })

    # Risk Attribution
    overall_port_std = np.sqrt(weights.T @ cov_mat @ weights)
    marginal_risk_contribution = (cov_mat @ weights) / overall_port_std
    component_risk_contribution = weights * marginal_risk_contribution
    pct_risk_contribution = component_risk_contribution / component_risk_contribution.sum()
    
    risk_attribution = pd.DataFrame({
        'Asset': n_assets,
        'Weight': weights,
        'Marginal Risk Contribution': marginal_risk_contribution,
        'Component Risk Contribution': component_risk_contribution,
        '% of Total Risk': pct_risk_contribution
    })


    return annual_results, risk_attribution

risk_free_rate = 0.02
n_trading_days = 252
tickers = ['AAPL', 'AMZN', 'GOOG', 'META', 'TSLA']
weights = [.20, .30, .80, .9, .9]
port_type = ['EW', 'MC', 'RP', 'OO']
confidence_level = 0.999

annual_results , risk_breakdown = portfolio_analytics(
    tickers=tickers, 
    risk_free_rate=risk_free_rate, 
    n_trading_days=n_trading_days, 
    confidence_level=confidence_level, 
    weights=weights, 
    port_type='CW')

print('Result')
print(annual_results)
risk_breakdown.sort_values('% of Total Risk', ascending=False)

[*********************100%***********************]  5 of 5 completed
[*********************100%***********************]  5 of 5 completed

Result
   Annual Port Return  Annual Port Std  Annual Port Sharpe  Annual VaR_0.999  \
0            0.265425         0.356288            0.688837          -1.67201   

   Annual ES_0.999  
0        -2.072185  


,Asset,Weight,Marginal Risk Contribution,Component Risk Contribution,% of Total Risk
TSLA,5,0.290323,0.033746,0.009797,0.436512
META,5,0.290323,0.021379,0.006207,0.276550
GOOG,5,0.258065,0.015411,0.003977,0.177202
AMZN,5,0.096774,0.016217,0.001569,0.069924
AAPL,5,0.064516,0.013850,0.000894,0.039812


In [85]:
results, risk_breakdown = portfolio_analytics(
    tickers=ticker_data['Ticker'].to_list(), 
    risk_free_rate=0.02,
    weights=ticker_data['Weight'].to_list(),
    confidence_level=0.975,
    port_type='CW', 
    n_trading_days=252)

print('Results')
print(results)
print(risk_breakdown)

[*********************100%***********************]  15 of 15 completed
[*********************100%***********************]  15 of 15 completed

Results
   Annual Port Return  Annual Port Std  Annual Port Sharpe  Annual VaR_0.975  \
0            0.256504         0.251748            0.939447         -0.475965   

   Annual ES_0.975  
0        -0.693265  
      Asset  Weight  Marginal Risk Contribution  Component Risk Contribution  \
NVDA     15   0.100                    0.011947                     0.001195   
TSLA     15   0.090                    0.014562                     0.001311   
AAPL     15   0.085                    0.011999                     0.001020   
BA       15   0.075                    0.018003                     0.001350   
XOM      15   0.070                    0.011976                     0.000838   
PLTR     15   0.065                    0.018654                     0.001213   
JPM      15   0.060                    0.009449                     0.000567   
DELL     15   0.060                    0.009287                     0.000557   
BLK      15   0.070                    0.022625                     0